# AI Tweet Detection: Dataset Selection and Merge

This notebook explains the datasets used for the AI tweet detection project and merges them into one clean dataset.

The datasets used are **TweepFake** and **ElectAI**.


## Dataset(s) used

The datasets used are **TweepFake** and **ElectAI**.

TweepFake gives the project a general human-vs-machine tweet dataset. ElectAI adds an election and civic-information angle, which makes the project more connected to a real-world use case. Using both datasets also makes the work stronger because the model is not trained on only one source.


## Source credibility

**TweepFake** comes from a published research project on deepfake tweet detection. It contains tweets from human accounts and bot accounts that were made to imitate those humans. The machine-generated tweets were produced using methods such as Markov Chains, RNN, LSTM, and GPT-2.

Source: https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0251415

**ElectAI** comes from an academic project on human-generated and AI-generated election claims in social media. It is useful for this project because it connects AI tweet detection to election and civic-information monitoring.

Source: https://arxiv.org/abs/2404.16116


## Label setup

The labels are converted into one binary format:

- `0` = human-written tweet
- `1` = AI-generated or machine-generated tweet

The original labels are kept in separate columns so the source of each label is still traceable.


In [ ]:
import pandas as pd

pd.set_option("display.max_colwidth", 120)


## Load TweepFake

This cell loads the TweepFake train, validation, and test files directly from the public GitHub repository.


In [ ]:
tweepfake_urls = {
    "train": "https://raw.githubusercontent.com/tizfa/tweepfake_deepfake_text_detection/master/data/splits/train.csv",
    "validation": "https://raw.githubusercontent.com/tizfa/tweepfake_deepfake_text_detection/master/data/splits/validation.csv",
    "test": "https://raw.githubusercontent.com/tizfa/tweepfake_deepfake_text_detection/master/data/splits/test.csv"
}

tweepfake_parts = []

for split_name, url in tweepfake_urls.items():
    temp = pd.read_csv(url, sep=";")
    temp["split"] = split_name
    tweepfake_parts.append(temp)

tweepfake_df = pd.concat(tweepfake_parts, ignore_index=True)

print("TweepFake rows:", len(tweepfake_df))
print("TweepFake columns:", tweepfake_df.columns.tolist())

display(tweepfake_df["account.type"].value_counts().to_frame("count"))
display(tweepfake_df["class_type"].value_counts().to_frame("count"))


TweepFake has **25,572 rows**. The labels are balanced between human-written and machine-generated tweets.


## Load ElectAI

This cell loads the ElectAI authorship-attribution train and test files directly from the public GitHub repository.


In [ ]:
electai_urls = {
    "train": "https://raw.githubusercontent.com/LanguageTechnologyLab/ElectAI/main/datasets/authorship%20attribution/train.csv",
    "test": "https://raw.githubusercontent.com/LanguageTechnologyLab/ElectAI/main/datasets/authorship%20attribution/test.csv"
}

electai_parts = []

for split_name, url in electai_urls.items():
    temp = pd.read_csv(url, sep="\t")
    temp["split"] = split_name
    electai_parts.append(temp)

electai_df = pd.concat(electai_parts, ignore_index=True)

print("ElectAI rows:", len(electai_df))
print("ElectAI columns:", electai_df.columns.tolist())

display(electai_df["label"].value_counts().to_frame("count"))


ElectAI has **9,900 rows**. The labels include human-written tweets and AI-generated tweets from models such as Falcon, Llama, and Mistral.


## Standardize the datasets

In [ ]:
tweepfake_clean = pd.DataFrame({
    "text": tweepfake_df["text"],
    "label": tweepfake_df["account.type"].map({"human": 0, "bot": 1}),
    "source_dataset": "TweepFake",
    "original_label": tweepfake_df["account.type"],
    "generator_type": tweepfake_df["class_type"],
    "split": tweepfake_df["split"]
})

electai_clean = pd.DataFrame({
    "text": electai_df["tweet"],
    "label": electai_df["label"].apply(lambda value: 0 if str(value).lower() == "human" else 1),
    "source_dataset": "ElectAI",
    "original_label": electai_df["label"],
    "generator_type": electai_df["label"],
    "split": electai_df["split"]
})

print("TweepFake standardized shape:", tweepfake_clean.shape)
print("ElectAI standardized shape:", electai_clean.shape)


Both datasets now use the same columns: `text`, `label`, `source_dataset`, `original_label`, `generator_type`, and `split`.


## Merge and clean

In [ ]:
combined_ai_tweets = pd.concat(
    [tweepfake_clean, electai_clean],
    ignore_index=True
)

rows_before_cleaning = len(combined_ai_tweets)

combined_ai_tweets = combined_ai_tweets.dropna(subset=["text", "label"]).copy()

combined_ai_tweets["text"] = (
    combined_ai_tweets["text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

combined_ai_tweets = combined_ai_tweets[combined_ai_tweets["text"] != ""]

rows_before_duplicates = len(combined_ai_tweets)

combined_ai_tweets["text_check"] = combined_ai_tweets["text"].str.lower().str.strip()

combined_ai_tweets = (
    combined_ai_tweets
    .drop_duplicates(subset="text_check")
    .drop(columns="text_check")
    .reset_index(drop=True)
)

rows_after_cleaning = len(combined_ai_tweets)

print("Rows before cleaning:", rows_before_cleaning)
print("Rows before duplicate removal:", rows_before_duplicates)
print("Final rows after cleaning:", rows_after_cleaning)
print("Duplicates removed:", rows_before_duplicates - rows_after_cleaning)


The cleaning is kept light because writing style matters for AI-text detection. The process removes missing text, empty text, extra spacing, and exact duplicate tweets.


## Final dataset summary

In [ ]:
source_summary = combined_ai_tweets["source_dataset"].value_counts().to_frame("row_count")
display(source_summary)

label_counts = combined_ai_tweets["label"].value_counts().sort_index()
label_percentages = (
    combined_ai_tweets["label"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

label_summary = pd.DataFrame({
    "label_name": ["human-written", "AI-generated"],
    "count": label_counts.values,
    "percentage": label_percentages.values
}, index=label_counts.index)

display(label_summary)

generator_summary = (
    combined_ai_tweets
    .groupby(["source_dataset", "generator_type"])
    .size()
    .reset_index(name="count")
)

display(generator_summary)


The final merged dataset has tweets from both TweepFake and ElectAI. The labels are standardized into human-written and AI-generated classes, while the source dataset and original generator labels are still kept for reference.


## Save merged dataset

In [ ]:
combined_ai_tweets.to_csv("combined_ai_tweet_detection_dataset.csv", index=False)

print("Saved combined dataset as: combined_ai_tweet_detection_dataset.csv")
print("Final dataset shape:", combined_ai_tweets.shape)
